## Load Data as Dataframe

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("powerplant_data.csv")
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [3]:
df.isna().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [4]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [5]:
X = df.drop("PE", axis=1)
y = df["PE"]

## Preprocess data

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [9]:
X_test_scaled

array([[ 1.34499288,  0.23869298, -1.28658067, -1.10532538],
       [ 0.81095912,  1.36269098, -0.74140656,  0.26485915],
       [-0.2437241 , -0.73900436,  1.99970178, -0.19713193],
       ...,
       [-0.67068342, -1.15902881, -0.29951077, -0.10651852],
       [ 1.31420898,  1.33752097, -0.87346737, -0.44288647],
       [-0.2611237 , -0.27021304,  0.37433797,  1.10646548]],
      shape=(1914, 4))

## Load data to Tensor

In [10]:
type(X_train_scaled)

numpy.ndarray

In [11]:
type(y_train)

pandas.core.series.Series

In [13]:
import torch
import torch.nn as nn

In [14]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float64)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float64)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float64).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float64).view(-1,1)

In [15]:
y_train_tensor

tensor([[442.7500],
        [432.5200],
        [428.8000],
        ...,
        [464.2600],
        [440.4500],
        [484.4400]], dtype=torch.float64)

In [16]:
from torch.utils.data import TensorDataset, DataLoader


In [61]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## Deep Learning

In [62]:
class ANN(nn.Module):
    def __init__(self):
        super (ANN, self).__init__()
        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),
    
            # 2nd hidden layer
            nn.Linear(6,6),
            nn.ReLU(),
    
            # output layer
            nn.Linear(6,1)
        )

    def forward(self, x):
        return self.model(x)

## Loss, Optimizer

In [63]:
import torch.optim as optim

model = ANN()
crietrion = nn.MSELoss()
optimizer = optim.Adam(model.parameters())

## Training The ANN

In [66]:
epochs = 10
train_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        # xb => features of 1 batch
        # yb => labels of 1 batch
        optimizer.zero_grad() # reset gradient to zero
        outputs = model(xb) # forward propagation => predicted outputs for this batch
        loss = crietrion(outputs, yb) # compute loss
        loss.backward() # backward propagation => computes gradients
        optimizer.step() # params update
        running_loss += loss.item() # loss is tensor, need to convert python float
        
        epoch_train_loss = running_loss / len(train_loader)
        train_lossess.append(epoch_train_loss)

RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float